# Avance 2. Ingeniería de características

## Objetivos
2.3 Crear nuevas características para mejorar el rendimiento de los modelos.

2.4 Mitigar el riesgo de características sesgadas y acelerar la convergencia de algunos algoritmos.

### Librerías y Carga de Datos

#### Librerías

In [1]:
import pandas as pd
import numpy as np
from tabulate import tabulate
from sklearn.preprocessing import OrdinalEncoder, OneHotEncoder, StandardScaler, MinMaxScaler, PowerTransformer
from sklearn.feature_selection import VarianceThreshold, SelectKBest, chi2, f_classif
from sklearn.decomposition import PCA
import seaborn as sns
import matplotlib.pyplot as plt

#### Carga de Datos

In [2]:
path = '../../data/processed/data.parquet'
df = pd.read_parquet(path)
df.head()

,Fecha,Cuadro,Estado,Codigo Padecimiento,Ax_003,Año,Semana,Ax_001,Ax_002,Valor,Padecimiento,Mes
0,2014-01-06,CUADRO 17.,Aguascalientes,F32,Sem.,2014,2,2014.0,0,1.0,DepresiónCIE-10ª REV.F32,1
1,2014-01-13,CUADRO 17.,Aguascalientes,F32,Sem.,2014,3,2014.0,0,2.0,DepresiónCIE-10ª REV.F32,1
2,2014-01-20,CUADRO 17.,Aguascalientes,F32,Sem.,2014,4,2014.0,0,1.0,DepresiónCIE-10ª REV.F32,1
3,2014-01-27,CUADRO 17.,Aguascalientes,F32,Sem.,2014,5,2014.0,0,1.0,DepresiónCIE-10ª REV.F32,1
4,2014-02-03,CUADRO 17.,Aguascalientes,F32,Sem.,2014,6,2014.0,0,2.0,DepresiónCIE-10ª REV.F32,2


#### Limpieza de Datos
Debido a que se cargó el dataset procesado del primer avance, los datos ya se encuentran limpios, sin embargo, como buena práctica y siguiendo la metodología CRISP-ML, nos aseguraremos de la limpieza

##### Asegurar que la columna Fecha sea de tipo datetime
Convertir a datetime permite manipular y extraer información temporal (año, mes, semana, trimestre)
y detectar posibles valores no válidos (se convierten en NaT).

In [3]:
print(df['Fecha'].dtypes)

datetime64[ns]


##### Estandarizar nombres de columnas
Estos nos permite unificar el formato de los nombres de columnas, facilitando su uso en código y evitando errores por inconsistencias.

In [4]:
df.columns = df.columns.str.strip()               # Elimina espacios al inicio o final
df.columns = df.columns.str.replace(' ', '_')     # Sustituye espacios por guiones bajos
df.columns = df.columns.str.lower()               # Convierte todo a minúsculas para consistencia

##### Verificar valores nulos
Identificar valores faltantes es crucial para decidir si se imputan, eliminan o se dejan como NaN (según su relevancia o proporción).

In [5]:
print('\nValores faltantes por columna:')
print(tabulate(df.isnull().sum().reset_index(),
               headers=['Columna', 'Valores faltantes'],
               tablefmt='rounded_outline',
               showindex=False))


Valores faltantes por columna:
╭─────────────────────┬─────────────────────╮
│ Columna             │   Valores faltantes │
├─────────────────────┼─────────────────────┤
│ fecha               │                   0 │
│ cuadro              │                   0 │
│ estado              │                   0 │
│ codigo_padecimiento │                   0 │
│ ax_003              │                   0 │
│ año                 │                   0 │
│ semana              │                   0 │
│ ax_001              │              509532 │
│ ax_002              │              509532 │
│ valor               │              509532 │
│ padecimiento        │                   0 │
│ mes                 │                   0 │
╰─────────────────────┴─────────────────────╯


In [6]:
df['valor'] = df['valor'].fillna(0).astype(int)

Por la naturaleza de ax_001 y ax_002 existen algunos valores nulos, esto no debería afectar al análisis, ya que funcionan como filtros estas columnas

##### Asegurar que 'valor' sea numérico
Forzamos a que 'valor' (número de casos) sea numérico. Si hay valores no convertibles (por error de digitación), se transforman en NaN para poder tratarlos más adelante.

In [7]:
print(df['valor'].dtypes)

int64


##### Comprobación general del tipo de datos

In [8]:
print("\nTipos de datos después de la limpieza:")
print(df.dtypes)


Tipos de datos después de la limpieza:
fecha                  datetime64[ns]
cuadro                         object
estado                         object
codigo_padecimiento            object
ax_003                         object
año                            UInt32
semana                         UInt32
ax_001                        float64
ax_002                         object
valor                           int64
padecimiento                   object
mes                             int32
dtype: object


#### Operaciones Comunes

##### Generación de nuevas características

Nota: en esta sección se esperaba agregar variables exógenas para mejorar el dataset y prepararlo para el futuro análisis, sin embargo, no se logró concretar una reunión de aprobación con la Dra. Ruth Pérez, líder de proyecto, por motivos de vacaciones laborales de la Dra. Es por esto, que a lo largo de este trabajo solo se trabajará con el dataset original, más la agregación de únicamente la variables exógenas que surgan del mismo dataset original.

###### Fechas de pandemia por COVID-19
Para analizar el efecto de la pandemia sobre los diagnósticos de depresión.

In [9]:
# Generar columna de temporada de COVID-19
# Rango de fechas de la pandemia en México
covid_start = pd.to_datetime('2020-03-01')
covid_end = pd.to_datetime('2022-03-31')

df['fecha'] = pd.to_datetime(df['fecha'])
df['covid'] = np.where((df['fecha'] >= covid_start) & (df['fecha'] <= covid_end), 1, 0)

df.head()

,fecha,cuadro,estado,codigo_padecimiento,ax_003,año,semana,ax_001,ax_002,valor,padecimiento,mes,covid
0,2014-01-06,CUADRO 17.,Aguascalientes,F32,Sem.,2014,2,2014.0,0,1,DepresiónCIE-10ª REV.F32,1,0
1,2014-01-13,CUADRO 17.,Aguascalientes,F32,Sem.,2014,3,2014.0,0,2,DepresiónCIE-10ª REV.F32,1,0
2,2014-01-20,CUADRO 17.,Aguascalientes,F32,Sem.,2014,4,2014.0,0,1,DepresiónCIE-10ª REV.F32,1,0
3,2014-01-27,CUADRO 17.,Aguascalientes,F32,Sem.,2014,5,2014.0,0,1,DepresiónCIE-10ª REV.F32,1,0
4,2014-02-03,CUADRO 17.,Aguascalientes,F32,Sem.,2014,6,2014.0,0,2,DepresiónCIE-10ª REV.F32,2,0


###### Temporada
Permite evaluar la influencia de factores climáticos o emocionales relacionados con las estaciones del año.

In [10]:
def asignar_temporada(mes):
    if mes in [12, 1, 2]:
        return 'Invierno'
    elif mes in [3, 4, 5]:
        return 'Primavera'
    elif mes in [6, 7, 8]:
        return 'Verano'
    else:
        return 'Otoño'

df['temporada'] = df['mes'].apply(asignar_temporada)

In [11]:
print('Dataset con nuevas características:')
df.head()

Dataset con nuevas características:


,fecha,cuadro,estado,codigo_padecimiento,ax_003,año,semana,ax_001,ax_002,valor,padecimiento,mes,covid,temporada
0,2014-01-06,CUADRO 17.,Aguascalientes,F32,Sem.,2014,2,2014.0,0,1,DepresiónCIE-10ª REV.F32,1,0,Invierno
1,2014-01-13,CUADRO 17.,Aguascalientes,F32,Sem.,2014,3,2014.0,0,2,DepresiónCIE-10ª REV.F32,1,0,Invierno
2,2014-01-20,CUADRO 17.,Aguascalientes,F32,Sem.,2014,4,2014.0,0,1,DepresiónCIE-10ª REV.F32,1,0,Invierno
3,2014-01-27,CUADRO 17.,Aguascalientes,F32,Sem.,2014,5,2014.0,0,1,DepresiónCIE-10ª REV.F32,1,0,Invierno
4,2014-02-03,CUADRO 17.,Aguascalientes,F32,Sem.,2014,6,2014.0,0,2,DepresiónCIE-10ª REV.F32,2,0,Invierno


##### Discretización (binning) de 'valor' en niveles bajo, medio y alto 
Agrupa los casos en categorías ordinales, útil para modelos interpretativos o análisis de tendencia.

In [12]:
df_nonzero = df[df['valor'] > 0]
df.loc[df['valor'] > 0, 'valor_cat'] = pd.qcut(df_nonzero['valor'], q=3, labels=['bajo', 'medio', 'alto'], duplicates='drop')
df.loc[df['valor'] == 0, 'valor_cat'] = 'bajo'

In [13]:
print('Dataset con nueva discretización:')
df.head()

Dataset con nueva discretización:


,fecha,cuadro,estado,codigo_padecimiento,ax_003,año,semana,ax_001,ax_002,valor,padecimiento,mes,covid,temporada,valor_cat
0,2014-01-06,CUADRO 17.,Aguascalientes,F32,Sem.,2014,2,2014.0,0,1,DepresiónCIE-10ª REV.F32,1,0,Invierno,bajo
1,2014-01-13,CUADRO 17.,Aguascalientes,F32,Sem.,2014,3,2014.0,0,2,DepresiónCIE-10ª REV.F32,1,0,Invierno,bajo
2,2014-01-20,CUADRO 17.,Aguascalientes,F32,Sem.,2014,4,2014.0,0,1,DepresiónCIE-10ª REV.F32,1,0,Invierno,bajo
3,2014-01-27,CUADRO 17.,Aguascalientes,F32,Sem.,2014,5,2014.0,0,1,DepresiónCIE-10ª REV.F32,1,0,Invierno,bajo
4,2014-02-03,CUADRO 17.,Aguascalientes,F32,Sem.,2014,6,2014.0,0,2,DepresiónCIE-10ª REV.F32,2,0,Invierno,bajo


##### Codificación (ordinal, one hot,…)

###### Codificación ordinal para variables con orden lógico
Se aplica codificación ordinal porque existe una jerarquía natural entre las categorías

In [14]:
# valor_cat
ordinal_vars = ['valor_cat']
encoder_ordinal = OrdinalEncoder()

df[ordinal_vars] = encoder_ordinal.fit_transform(df[ordinal_vars])

###### Codificación one-hot para variables nominales (sin orden)
La codificación one-hot convierte categorías no ordenadas en variables binarias (dummy variables),
evitando que el modelo interprete jerarquías inexistentes y reduciendo colinealidad con drop_first=True.

In [15]:
nominal_vars = ['estado', 'temporada', 'codigo_padecimiento']
df = pd.get_dummies(df, columns=nominal_vars, drop_first=True)

In [16]:
print('Dataset con nueva codificaciones:')
df.head()

Dataset con nueva codificaciones:


,fecha,cuadro,ax_003,año,semana,ax_001,ax_002,valor,padecimiento,mes,...,codigo_padecimiento_G20,codigo_padecimiento_G30,codigo_padecimiento_R45.8,codigo_padecimiento_X60,codigo_padecimiento_X61,codigo_padecimiento_X70,codigo_padecimiento_X72,codigo_padecimiento_X78,codigo_padecimiento_X80,codigo_padecimiento_Z91.5
0,2014-01-06,CUADRO 17.,Sem.,2014,2,2014.0,0,1,DepresiónCIE-10ª REV.F32,1,...,False,False,False,False,False,False,False,False,False,False
1,2014-01-13,CUADRO 17.,Sem.,2014,3,2014.0,0,2,DepresiónCIE-10ª REV.F32,1,...,False,False,False,False,False,False,False,False,False,False
2,2014-01-20,CUADRO 17.,Sem.,2014,4,2014.0,0,1,DepresiónCIE-10ª REV.F32,1,...,False,False,False,False,False,False,False,False,False,False
3,2014-01-27,CUADRO 17.,Sem.,2014,5,2014.0,0,1,DepresiónCIE-10ª REV.F32,1,...,False,False,False,False,False,False,False,False,False,False
4,2014-02-03,CUADRO 17.,Sem.,2014,6,2014.0,0,2,DepresiónCIE-10ª REV.F32,2,...,False,False,False,False,False,False,False,False,False,False


##### Escalamiento (normalización, estandarización, min – max,…)

In [17]:
scaler_minmax = MinMaxScaler()
df[['valor_minmax']] = scaler_minmax.fit_transform(df[['valor']])

In [18]:
print('Dataset con escalamiento:')
df.head()

Dataset con escalamiento:


,fecha,cuadro,ax_003,año,semana,ax_001,ax_002,valor,padecimiento,mes,...,codigo_padecimiento_G30,codigo_padecimiento_R45.8,codigo_padecimiento_X60,codigo_padecimiento_X61,codigo_padecimiento_X70,codigo_padecimiento_X72,codigo_padecimiento_X78,codigo_padecimiento_X80,codigo_padecimiento_Z91.5,valor_minmax
0,2014-01-06,CUADRO 17.,Sem.,2014,2,2014.0,0,1,DepresiónCIE-10ª REV.F32,1,...,False,False,False,False,False,False,False,False,False,0.000006
1,2014-01-13,CUADRO 17.,Sem.,2014,3,2014.0,0,2,DepresiónCIE-10ª REV.F32,1,...,False,False,False,False,False,False,False,False,False,0.000013
2,2014-01-20,CUADRO 17.,Sem.,2014,4,2014.0,0,1,DepresiónCIE-10ª REV.F32,1,...,False,False,False,False,False,False,False,False,False,0.000006
3,2014-01-27,CUADRO 17.,Sem.,2014,5,2014.0,0,1,DepresiónCIE-10ª REV.F32,1,...,False,False,False,False,False,False,False,False,False,0.000006
4,2014-02-03,CUADRO 17.,Sem.,2014,6,2014.0,0,2,DepresiónCIE-10ª REV.F32,2,...,False,False,False,False,False,False,False,False,False,0.000013


##### Transformación (logarítmica, exponencial, raíz cuadrada, Box – Cox, Yeo – Johnson,…)

La transformación Yeo–Johnson es adecuada para valores positivos y negativos, y corrige asimetrías sin requerir logaritmos, lo que mejora el rendimiento de modelos sensibles a la normalidad.

In [19]:
# Transformación de potencia (Yeo–Johnson) para normalizar distribución
pt = PowerTransformer(method='yeo-johnson')
df[['valor_yeojohnson']] = pt.fit_transform(df[['valor']])

In [20]:
print('Dataset con transformación Yeo-Johnson:')
df.head()

Dataset con transformación Yeo-Johnson:


,fecha,cuadro,ax_003,año,semana,ax_001,ax_002,valor,padecimiento,mes,...,codigo_padecimiento_R45.8,codigo_padecimiento_X60,codigo_padecimiento_X61,codigo_padecimiento_X70,codigo_padecimiento_X72,codigo_padecimiento_X78,codigo_padecimiento_X80,codigo_padecimiento_Z91.5,valor_minmax,valor_yeojohnson
0,2014-01-06,CUADRO 17.,Sem.,2014,2,2014.0,0,1,DepresiónCIE-10ª REV.F32,1,...,False,False,False,False,False,False,False,False,0.000006,0.283624
1,2014-01-13,CUADRO 17.,Sem.,2014,3,2014.0,0,2,DepresiónCIE-10ª REV.F32,1,...,False,False,False,False,False,False,False,False,0.000013,0.662544
2,2014-01-20,CUADRO 17.,Sem.,2014,4,2014.0,0,1,DepresiónCIE-10ª REV.F32,1,...,False,False,False,False,False,False,False,False,0.000006,0.283624
3,2014-01-27,CUADRO 17.,Sem.,2014,5,2014.0,0,1,DepresiónCIE-10ª REV.F32,1,...,False,False,False,False,False,False,False,False,0.000006,0.283624
4,2014-02-03,CUADRO 17.,Sem.,2014,6,2014.0,0,2,DepresiónCIE-10ª REV.F32,2,...,False,False,False,False,False,False,False,False,0.000013,0.662544


#### Selección y extracción de características


###### Eliminación de columnas no necesarias
Después de las transformaciones, escalamientos y codificaciones, podemos eliminar algunas columnas que ya no serán necesarias para el análisis

In [21]:
df_analysis = df.drop(columns=['cuadro', 'padecimiento', 'valor_cat'])
df_analysis.head()

,fecha,ax_003,año,semana,ax_001,ax_002,valor,mes,covid,estado_Baja California,...,codigo_padecimiento_R45.8,codigo_padecimiento_X60,codigo_padecimiento_X61,codigo_padecimiento_X70,codigo_padecimiento_X72,codigo_padecimiento_X78,codigo_padecimiento_X80,codigo_padecimiento_Z91.5,valor_minmax,valor_yeojohnson
0,2014-01-06,Sem.,2014,2,2014.0,0,1,1,0,False,...,False,False,False,False,False,False,False,False,0.000006,0.283624
1,2014-01-13,Sem.,2014,3,2014.0,0,2,1,0,False,...,False,False,False,False,False,False,False,False,0.000013,0.662544
2,2014-01-20,Sem.,2014,4,2014.0,0,1,1,0,False,...,False,False,False,False,False,False,False,False,0.000006,0.283624
3,2014-01-27,Sem.,2014,5,2014.0,0,1,1,0,False,...,False,False,False,False,False,False,False,False,0.000006,0.283624
4,2014-02-03,Sem.,2014,6,2014.0,0,2,2,0,False,...,False,False,False,False,False,False,False,False,0.000013,0.662544


Las estrategias clásicas de extracción de características en este caso específico, consideramos, no serían adecuadas debido a que los datos no son continuos ni correlacionables, hay poca variabilidad en las columnas, la estructura es categórica y administrativa, falta un volumen de características numéricas suficientes.

In [ ]:
# # Guarda el df_analysis en un archivo parquet
# df_analysis.to_parquet('../../data/interim/data.parquet')
# print(f'DataFrame guardado exitosamente con {len(df_analysis):,} registros')
# print(f'Archivo guardado en: ../../data/interim/data.parquet')

DataFrame guardado exitosamente con 839,169 registros
Archivo guardado en: ../../data/interim/data.parquet


#### Conclusiones

Durante la fase de Preparación de los datos, correspondiente a la metodología CRISP-ML, se llevaron a cabo las tareas necesarias para garantizar la calidad, consistencia y pertinencia del conjunto de datos utilizado en el análisis. Este proceso resultó fundamental para asegurar que la información recopilada fuera adecuada para el posterior modelado y la obtención de resultados confiables.

En primer lugar, se realizó una inspección estructural del dataset, identificándose variables con nombres inconsistentes, presencia de caracteres no reconocidos y valores faltantes en columnas clave, particularmente en la variable Valor. Se implementaron procedimientos de limpieza y normalización básica, orientados a estandarizar los nombres de las columnas, eliminar o imputar datos vacíos, y transformar los valores no numéricos en formatos analizables.

El análisis exploratorio evidenció además que la información disponible tiene un carácter administrativo y acumulativo, derivado de reportes epidemiológicos más que de mediciones individuales. Por ello, se consideró que las transformaciones de reducción de dimensionalidad no aportarían beneficios significativos, dado que las variables no presentan relaciones lineales o numéricas que justifiquen su aplicación.

En general, la fase de preparación permitió depurar, comprender y estructurar adecuadamente los datos, sentando las bases para un modelado coherente con la naturaleza del problema y las características de la información. Si bien las limitaciones del dataset restringen el uso de ciertas técnicas de ingeniería de características, el proceso permitió identificar los desafíos inherentes al tratamiento de datos de vigilancia en salud pública, reforzando la importancia de la curación y contextualización de los datos en proyectos de inteligencia artificial aplicada al ámbito epidemiológico.